# Data Preprocessing - AQI Dataset

## 1. Load the dataset

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

data_path = '../data/raw/station_day.csv'
if not os.path.exists(data_path):
    data_path = 'data/raw/station_day.csv'
features = [
    'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3',
    'CO', 'SO2', 'O3', 'Benzene', 'Toluene'
]

data = pd.read_csv(data_path)
print('Original shape:', data.shape)

Original shape: (97335, 16)


## 2. Clean the data

In [2]:
data = data.dropna(subset=['AQI', 'AQI_Bucket'])
data = data.drop_duplicates(subset=features + ['AQI', 'AQI_Bucket'])
data['Date'] = pd.to_datetime(data['Date'])
data = data.sort_values(['Date', 'StationId']).reset_index(drop=True)

print('Shape after cleaning:', data.shape)
data.head()

Shape after cleaning: (77254, 16)


,StationId,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket
0,DL007,2015-01-01,425.49,607.98,49.37,40.91,90.29,NaN,0.69,NaN,143.84,9.19,14.16,9.84,579.0,Severe
1,DL013,2015-01-01,200.96,NaN,17.06,12.39,24.73,42.98,21.82,8.91,NaN,NaN,NaN,NaN,364.0,Very Poor
2,DL007,2015-01-02,172.13,269.55,22.97,40.98,63.94,NaN,1.35,NaN,77.99,4.01,6.19,4.29,497.0,Severe
3,DL013,2015-01-02,251.12,NaN,26.02,17.12,31.50,46.70,13.29,6.79,NaN,NaN,NaN,NaN,410.0,Severe
4,DL007,2015-01-03,84.42,131.90,10.17,40.89,51.06,NaN,1.54,NaN,57.48,1.86,2.87,1.99,141.0,Moderate


In [3]:
X = data[features]
y = data['AQI']

print('Input columns:', X.columns.tolist())
print('Output column: AQI')

Input columns: ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene']
Output column: AQI


## 3. Split training and testing data

In [4]:
dates = data['Date'].drop_duplicates()
train_dates, test_dates = train_test_split(
    dates, test_size=0.2, random_state=42
)

train_data = data[data['Date'].isin(train_dates)]
test_data = data[data['Date'].isin(test_dates)]

X_train = train_data[features]
X_test = test_data[features]
y_train = train_data['AQI']
y_test = test_data['AQI']

print('Training rows:', len(X_train))
print('Testing rows:', len(X_test))

Training rows: 61404
Testing rows: 15850


## 4. Handle missing values

In [5]:
print('Missing values before filling:')
print(X_train.isnull().sum())

imputer = SimpleImputer(strategy='median')
X_train_filled = imputer.fit_transform(X_train)
X_test_filled = imputer.transform(X_test)

print()
print('Missing values after filling:', np.isnan(X_train_filled).sum())

Missing values before filling:
PM2.5       3432
PM10       18440
NO          2347
NO2          882
NOx         3396
NH3        22003
CO          4396
SO2         7579
O3          6911
Benzene    15849
Toluene    19849
dtype: int64

Missing values after filling: 0


## 5. Scale data for KNN and SVR

In [6]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_filled)
X_test_scaled = scaler.transform(X_test_filled)

print('Scaled training shape:', X_train_scaled.shape)
print('Scaled testing shape:', X_test_scaled.shape)

Scaled training shape: (61404, 11)
Scaled testing shape: (15850, 11)


## 6. Preprocessing used by the models

In [7]:
knn_svr_preprocessing = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

tree_preprocessing = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

print('KNN and SVR: median filling + scaling')
print('Random Forest and XGBoost: median filling')

KNN and SVR: median filling + scaling
Random Forest and XGBoost: median filling
